In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from datetime import datetime
from zoneinfo import ZoneInfo

In [2]:
# Step 1: Load the Dataset
apps_df = pd.read_csv(r"C:\Users\DELL\Pictures\elevance_skills\play_store_data\apps.csv")
reviews_df = pd.read_csv(r"C:\Users\DELL\Pictures\elevance_skills\play_store_data\user_reviews.csv")

In [3]:
area_data = apps_df.copy()

area_data["Last Updated"] = pd.to_datetime(
    area_data["Last Updated"],
    errors="coerce"
)

area_data = area_data.dropna(
    subset=["Last Updated"]
)

In [4]:
area_data = area_data[
    area_data["Rating"] >= 4.2
]

area_data = area_data[
    area_data["Reviews"] > 1000
]

In [5]:
area_data["App"] = area_data[
    "App"
].astype(str)

area_data = area_data[
    ~area_data["App"]
    .str.contains(
        r"\d",
        regex=True,
        na=False
    )
]

In [6]:
area_data["Category"] = area_data[
    "Category"
].astype(str)

area_data = area_data[
    area_data["Category"]
    .str.upper()
    .str.startswith(
        ("T", "P")
    )
]

In [7]:
area_data["Size"] = pd.to_numeric(
    area_data["Size"],
    errors="coerce"
)

area_data = area_data.dropna(
    subset=["Size"]
)

area_data = area_data[
    area_data["Size"].between(
        20,
        80
    )
]

In [8]:
category_translation = {

    "TRAVEL_AND_LOCAL": "Voyages et local",

    "PRODUCTIVITY": "Productividad",

    "PHOTOGRAPHY": "写真"
}


area_data["Graph_Category"] = (
    area_data["Category"]
    .str.upper()
    .replace(
        category_translation
    )
)

In [9]:
area_data["Installs"] = (
    area_data["Installs"]
    .astype(str)
    .str.replace(
        ",",
        "",
        regex=False
    )
    .str.replace(
        "+",
        "",
        regex=False
    )
)

area_data["Installs"] = pd.to_numeric(
    area_data["Installs"],
    errors="coerce"
)

area_data = area_data.dropna(
    subset=["Installs"]
)


# Create Month column

area_data["Month"] = (
    area_data["Last Updated"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

In [10]:
# Total installs per month and category

monthly_data = (
    area_data
    .groupby(
        [
            "Month",
            "Graph_Category"
        ]
    )["Installs"]
    .sum()
    .reset_index()
)


# Convert to table format

stacked_data = (
    monthly_data
    .pivot(
        index="Month",
        columns="Graph_Category",
        values="Installs"
    )
    .fillna(0)
)


# Sort by month

stacked_data = stacked_data.sort_index()


# Calculate cumulative installs

cumulative_data = (
    stacked_data.cumsum()
)


# Calculate month-over-month growth

growth_data = (
    stacked_data
    .pct_change()
    * 100
)


cumulative_data.head()

Graph_Category,PARENTING,PERSONALIZATION,Productividad,TOOLS,Voyages et local,写真
Month,,,,,,
2014-11-01,0.0,0.0,0.0,0.0,0.0,1000000.0
2016-10-01,0.0,0.0,1000000.0,0.0,0.0,1000000.0
2016-12-01,0.0,1000000.0,1000000.0,0.0,0.0,1000000.0
2017-03-01,0.0,1000000.0,1000000.0,0.0,0.0,51000000.0
2017-06-01,0.0,1000000.0,1000000.0,0.0,0.0,61000000.0


In [11]:
# Get current IST time

india_time = datetime.now(
    ZoneInfo("Asia/Kolkata")
)

current_hour = india_time.hour


# Show chart only between 4 PM and 6 PM IST

if 16 <= current_hour < 18:


    # Create figure

    fig, ax = plt.subplots(
        figsize=(14, 7)
    )


    # Get months

    months = cumulative_data.index


    # Get category names

    categories = cumulative_data.columns


    # Create stacked area chart

    ax.stackplot(
        months,
        *[
            cumulative_data[category]
            for category in categories
        ],
        labels=categories,
        alpha=0.6
    )


    # Find months where any category
    # increased by more than 25%

    significant_months = growth_data.index[
        (
            growth_data > 25
        ).any(
            axis=1
        )
    ]


    # Highlight significant growth months

    for month in significant_months:

        ax.axvspan(
            month - pd.Timedelta(days=15),
            month + pd.Timedelta(days=15),
            alpha=0.15
        )


    # Title

    ax.set_title(
        "Cumulative Number of Installs Over Time by App Category"
    )


    # X-axis

    ax.set_xlabel(
        "Month"
    )


    # Y-axis

    ax.set_ylabel(
        "Cumulative Installs"
    )


    # Legend

    ax.legend(
        title="App Category",
        loc="upper left"
    )


    # Rotate dates

    plt.xticks(
        rotation=45
    )


    plt.tight_layout()


    # Display graph below the Jupyter cell

    plt.show()


else:

    print(
        "This visualization is available only "
        "between 4 PM IST and 6 PM IST."
    )

This visualization is available only between 4 PM IST and 6 PM IST.
